# 5 The notebook as handed out, and a study of your own, on Optuna

Notebook 04 ran one search and read it for you. Here you drive, in two parts.

Every search here is an Optuna study, stored in `artefacts/optuna/studies.db` and read in
optuna-dashboard (`make optuna-dashboard`); every run you hand in is one trial of it.

**Part 1 is the notebook as handed out.** A baseline with its spread between runs, the encoder
comparison, one search over a given space, its winner rerun a few times, and a noise
ablation. You change nothing in it. Nothing in it is seeded: every model starts from other
random weights, on your machine and on everyone else's, so your numbers will not be your
neighbour's. That is the point. Fifteen baselines side by side show how far one model
fluctuates from run to run, and so how large a difference has to be before it means
anything. The search samples at random as well: every student draws other trials from the
same space, so fifteen searches side by side show how much of a search's winner is the
draw, and pooled they are one search of the space fifteen times the size.

**Part 2 is your own study.** You have an idea you want to test, design the search space
that tests it, give the study a name, and run it. Every run of the study carries its name
and its search space, so what reaches the next lesson is the runs themselves: two sets per
student, the handed-out ones and your study. A study that disappointed is a result worth
bringing. "I expected X, measured Y, and here is the spread between runs that makes me believe it"
is the habit this lesson exists for, whichever way it came out.

Everything starts on **DTD**: 47 texture classes, roughly twenty points of headroom, and
an encoder axis that actually moves.

In [ ]:
import sys
import time
from functools import cache

import datasets
import numpy as np
import optuna
import pandas as pd
import torch
from loguru import logger
from notebooktester import param
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.components import (
    Gate,
    GaussianNoise,
    NeuralNet,
    Norm,
    Projection,
    Serial,
    Skip,
)
from vectormesh.data import Collate, OneHot

from hpo.architecture import Architecture
from hpo.budget import Budget
from hpo.ideas import Idea, study_name
from hpo.metrics import Repeats
from hpo.runs import RunRecord, StudyLog
from hpo.space import Choice, Float, Int, Space
from hpo.store import Store
from hpo.training import Fit, Training, repeat

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
# a search shows one progress bar; Optuna's line per trial stays off
optuna.logging.set_verbosity(optuna.logging.WARNING)

EPOCHS = param(15, test=1)
N_TRIALS = param(24, test=2)
REPEATS = param(3, test=2)

DATASET = "dtd"
ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

# Every published cache, by dataset and encoder. A study on another dataset or encoder
# picks its cache from here.
CACHES = {
    "dtd": {
        enc: f"pttrn-io/dtd-{enc}"
        for enc in [
            "dinov2-small",
            "dinov2-base",
            "dinov2-large",
            "resnet-18",
            "mobilenet_v2_1.0_224",
        ]
    },
    "eurosat": {enc: f"pttrn-io/eurosat-{enc}" for enc in ENCODERS},
    "oxford-flowers": {enc: f"pttrn-io/oxford-flowers-{enc}" for enc in ENCODERS},
    "fashion-mnist": {
        enc: f"pttrn-io/fashion-mnist-{enc}"
        for enc in ["dinov2-small", "mobilenet_v2_1.0_224"]
    },
}

# The values a string param can take, for the runs that are given their params (see
# hpo.runs.StudyLog).
CHOICES = {
    "dataset": list(CACHES),
    "encoder": sorted({enc for encoders in CACHES.values() for enc in encoders}),
    "activation": ["relu", "gelu", "tanh", "sigmoid", "none"],
    "norm": ["none", "batch", "layer"],
    "arm": ["control", "treatment"],
}

# Part 1 keeps two Optuna studies: `lesson1-notebook`, the trials of the handed-out
# search, and `lesson1-notebook-runs`, every run that is given its params (the
# baseline, the encoder comparison, the winner, the noise ablation). Part 2 keeps the
# same pair under your study's name.
# Both carry the course's idea: the lesson as handed out, the same for everyone.
# The store says where the studies, the artifacts and the trainer's logs live.
STORE = Store.default()
HANDED_OUT = Idea.handed_out(1)
RUNS = StudyLog.runs("lesson1-notebook-runs", HANDED_OUT, choices=CHOICES)

In [ ]:
def load(dataset: str, encoder: str, split: str) -> VectorCache:
    if encoder not in CACHES.get(dataset, {}):
        raise KeyError(
            f"no published cache for {dataset!r} with {encoder!r}; "
            f"CACHES has { ({d: list(e) for d, e in CACHES.items()}) }"
        )
    return VectorCache.from_hub(CACHES[dataset][encoder], split=split)


@cache
def shape_of(dataset: str, encoder: str) -> tuple[int, int]:
    """(vector width, number of classes) of a cache."""
    train = load(dataset, encoder, "train")
    return train.dim, train.num_classes


def loaders(
    dataset: str, encoder: str, batch_size: int = 64, n_train: int | None = None
):
    # n_train shrinks the training set for the data-size questions below.
    # The shuffle is NOT optional -- see the cell after this one.
    out = []
    for split in ("train", "test"):
        cache = load(dataset, encoder, split)
        ds = cache.dataset
        if n_train is not None and split == "train":
            ds = ds.shuffle(seed=0).select(range(n_train))
        oh = OneHot(
            num_classes=cache.num_classes, label_col="label", target_col="onehot"
        )
        col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
        out.append(
            DataLoader(
                ds.map(oh),
                batch_size=batch_size,
                shuffle=(split == "train"),
                collate_fn=col,
            )
        )
    return out


def classes_of(loader) -> list[str]:
    """The names of the classes of a loader's cache, in the order of their labels."""
    return list(loader.dataset.features["label"].names)


def run(
    model,
    train_dl,
    test_dl,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> Fit:
    """Train `model` and return the fit: the confusion on the test set after the last
    epoch, the seconds spent in training steps, and the curves. `fit.scores()` holds
    every metric: accuracy, macro recall, and recall, precision and F1 per class
    (`hpo.metrics`); every run logs them all."""
    # hpo.training.Training is vectormesh's Trainer, timed, and scored on the whole test
    # set after every epoch; progress=False inside it drops the tqdm bars, which is what
    # makes calling this once per run -- several times per exercise below -- cheap.
    training = Training(lr=lr, epochs=epochs, logdir=STORE.logdir())
    return training.fit(model, train_dl, test_dl, classes_of(test_dl))


# Every choice a head can make, with the value that leaves it out. A run logs all of them
# (see run_params), so each is an axis in the class's parallel-coordinates plot.
HEAD = {
    "n_hidden_layers": 0,
    "hidden_dim": 0,
    "activation": "gelu",
    "sigma": 0.0,
    "norm": "none",
    "dropout": 0.0,
    "skip": False,
    "gate": False,
    "lr": 1e-3,
}

ACTIVATIONS = {"gelu": nn.GELU, "relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


def make_head(config: dict, dim: int, n_classes: int) -> Serial:
    # the same builder as notebook 04 §4.1: 0 hidden layers is the baseline's single Linear,
    # more layers with the defaults go through one NeuralNet call, which puts a GELU between
    # every Linear. activation (a key of ACTIVATIONS), norm ("batch", "layer" or "none"),
    # dropout (a probability), skip and gate change the hidden block; they do nothing
    # without hidden layers. lr goes to the optimizer, not here. Add an axis to your
    # param_space? Use it here, and give it a default in HEAD.
    c = {**HEAD, **config}
    n_hidden, width = c["n_hidden_layers"], c["hidden_dim"]
    if n_hidden == 0:
        head = nn.Linear(dim, n_classes)
    elif (
        c["activation"] == "gelu"
        and c["norm"] == "none"
        and c["dropout"] == 0
        and not c["skip"]
        and not c["gate"]
    ):
        head = NeuralNet([dim, *[width] * n_hidden], n_classes)
    else:
        layers, d = [], dim
        for _ in range(n_hidden):
            layers.append(nn.Linear(d, width))
            if c["norm"] != "none":
                layers.append(Norm(width, kind=c["norm"]))
            layers.append(ACTIVATIONS[c["activation"]]())
            if c["dropout"] > 0:
                layers.append(nn.Dropout(c["dropout"]))
            d = width
        block = Serial(layers)
        if c["skip"]:  # transform(norm(x)) + projection(norm(x)), see vectormesh's Skip
            block = Skip(block, in_size=dim, projection=Projection(dim, width))
        head = Serial(
            [block, *([Gate(width)] if c["gate"] else []), nn.Linear(width, n_classes)]
        )
    return Serial([GaussianNoise(sigma=c["sigma"]), head])


def run_params(dataset: str, encoder: str, config: dict | None = None, **extra) -> dict:
    """The params of a run: the vectors it starts from, every head choice, the budget.

    The head choices are the ones the model was built with: without hidden layers there
    is no hidden block, so hidden_dim, activation, norm, dropout, skip and gate are logged
    as off whatever the config drew."""
    head = {
        **HEAD,
        **{k: v for k, v in (config or {}).items() if k not in ("dataset", "encoder")},
    }
    if head["n_hidden_layers"] == 0:
        head.update(
            hidden_dim=0,
            activation="none",
            norm="none",
            dropout=0.0,
            skip=False,
            gate=False,
        )
    return {
        "dataset": dataset,
        "encoder": encoder,
        "embed_dim": shape_of(dataset, encoder)[0],
        "epochs": EPOCHS,
        **head,
        **extra,
    }


def run_spread(
    make_model, train_dl, test_dl, repeats: int = REPEATS, **kwargs
) -> Repeats:
    """Every metric of `repeats` runs, each from other random weights (nothing is
    seeded), and the mean seconds spent in training steps. `.values("accuracy")` takes
    one metric of every repeat."""
    return repeat(
        make_model, lambda model: run(model, train_dl, test_dl, **kwargs), repeats
    )


def compare(name: str, scores: np.ndarray, versus: np.ndarray | None = None) -> None:
    line = f"{name:<32} {scores.mean():.4f} +/- {scores.std():.4f}   (n={len(scores)})"
    if versus is not None:
        diff = scores.mean() - versus.mean()
        pooled = np.sqrt(
            scores.var(ddof=1) / len(scores) + versus.var(ddof=1) / len(versus)
        )
        line += f"   {100 * diff:+.2f} pt, {diff / pooled:+.1f} sigma"
    print(line)


DIM = {enc: shape_of(DATASET, enc)[0] for enc in ENCODERS}
N_CLASSES = shape_of(DATASET, "dinov2-small")[1]
print("widths:", DIM, "| classes:", N_CLASSES)

# Part 1 — the notebook as handed out

### First, look at the data you are about to slice

That `.shuffle(seed=0)` above is not defensive habit, it is load-bearing, and the reason is
worth two minutes because it is the kind of bug that produces a *plausible* wrong answer
rather than a crash.

In [ ]:
dtd = load(DATASET, "dinov2-small", "train")
labels = dtd["label"][:]
shuffled_labels = dtd.dataset.shuffle(seed=0)["label"][:]

print("first 20 labels in the DTD cache:", labels[:20].tolist())
print()
print(f"{'rows taken':>12} {'classes, as stored':>20} {'classes, shuffled':>19}")
for n in (470, 940, 1880):
    print(
        f"{n:>12} {len(set(labels[:n].tolist())):>14}/{N_CLASSES:<5} "
        f"{len(set(shuffled_labels[:n].tolist())):>13}/{N_CLASSES}"
    )

**The DTD cache is stored grouped by label.** Take the first 940 rows and you do not get a
smaller sample of the task — you get a *different, easier* task with a quarter of the
classes, and a test set full of categories your model has never once seen.

The reason this is dangerous rather than merely wrong is that it fails *quietly and
plausibly*. Subsampling this way at 470 / 940 / 1880 rows produces test accuracies of
0.12 / 0.24 / 0.44, which looks exactly like a healthy learning curve — right up until you
notice those numbers are just `n / 3760`. The model was scoring near-perfectly on the
classes it had and zero on the rest. Shuffled, the same sizes give about 0.69 / 0.75 / 0.78
(measured with a one-hidden-layer head; `planning/evidence.md` §3b), which
is the real curve and a far less exciting one.

Two general lessons, both cheap:

- **Not every cache is shuffled.** The EuroSAT caches are (they were subsampled at build
  time, which shuffles); the DTD ones were built `--full`, which does not. Do not assume —
  it is one line to check, as above.
- **A number that looks like a textbook result deserves more suspicion, not less.** The
  broken version of this exercise produced a *nicer* curve than the correct one.

## E1 — The baseline, first

**Before you search anything**, establish the two-line baseline on DTD and its spread
between runs. A search without a baseline is unreadable; a baseline without a spread cannot tell
you whether a later "win" is real.

Every number you produce below gets compared against these two.

In [ ]:
train_dl, test_dl = loaders(DATASET, "dinov2-small")

baseline_runs = run_spread(
    lambda: nn.Linear(DIM["dinov2-small"], N_CLASSES), train_dl, test_dl
)
baseline = baseline_runs.values("accuracy")
compare("baseline: Linear(384, 47)", baseline)
print(
    f"\n=> a later result must clear {baseline.mean() + 2 * baseline.std():.4f} "
    f"before you should believe it beat this."
)

example = next(iter(train_dl))[0][
    :2
]  # two rows are enough to trace every layer's shape
RUNS.log(
    "baseline",
    "baseline",
    baseline_runs,
    dataset=DATASET,
    params=run_params(DATASET, "dinov2-small"),
    architecture=Architecture.of(nn.Linear(DIM["dinov2-small"], N_CLASSES), example),
)

Both numbers are now in the Optuna store, written before anything else ran. That order is the point:
a baseline fixed in advance is the only thing that stops you from quietly redefining what
counted as success once you have seen the results.

## E2 — Argue from the metadata, then check

Notebook 01 said the metadata is the public API of a cache. This is not a cold guess —
you already have evidence, so use it.

**Before running the next cell**, answer for yourself:

1. Look back at notebook 01 §1.5's table: DINOv2 beat resnet-18 and mobilenet by 9-15
   points on *flowers*. Textures are a different domain, but the *ranking* of general-purpose
   image encoders tends to be fairly stable across natural-image tasks. Would you expect the
   same ordering here, or is there a reason textures specifically might reshuffle it? Either
   answer is defensible — the point is to have reasoned about it, not to have guessed right.
2. `mobilenet_v2` produces 1280 numbers per image against DINOv2's 384. Notebook 01 §1.5
   already told you width is not quality on flowers (`mobilenet_v2` lost by eleven points
   there despite more dimensions). Do you expect that to hold here too?
3. On flowers the encoder spread was about 15 points and on EuroSAT about 3. DTD's spread is
   somewhere in that range — pick a number and say why textures should be closer to one end
   than the other.

Then run it.

In [ ]:
results = {}
for enc in ENCODERS:
    tr, te = loaders(DATASET, enc)
    fit = run(nn.Linear(DIM[enc], N_CLASSES), tr, te)
    results[enc] = fit.scores().accuracy
    RUNS.log(
        f"encoder-{enc}",
        "ablation",
        fit.as_repeats(),
        dataset=DATASET,
        params=run_params(DATASET, enc),
        architecture=Architecture.of(
            nn.Linear(DIM[enc], N_CLASSES), torch.zeros(2, DIM[enc])
        ),
    )

print(f"{'encoder':<24} {'width':>6} {'accuracy':>9}")
print("-" * 42)
for enc, acc in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"{enc:<24} {DIM[enc]:>6} {acc:>9.4f}")

spread = max(results.values()) - min(results.values())
print(f"\nspread across encoders: {100 * spread:.1f} pt")
print(
    f"for comparison, the baseline's own spread between runs: {100 * baseline.std():.2f} pt"
)

Compare that last pair of numbers. **How many times the spread between runs is the encoder axis wide?** That
ratio is the honest way to say "this axis matters and that one does not".

## E3 — The handed-out search

The search below is the one handed out to the whole class, on `dinov2-small` only. Its
axes are the learning goals of this lesson (`references/leerdoelen/les1.md`):

| axis | values | the idea it tests | goal |
|---|---|---|---|
| `n_hidden_layers` | 0 to 3 | depth; 0 is the linear baseline | 1.2 |
| `hidden_dim` | 32 to 1024 | width: more units, more linear pieces | 1.7 |
| `activation` | ReLU, GELU, tanh, sigmoid | the nonlinearity, and saturation | 1.5 |
| `norm` | none, batch, layer | normalisation in the hidden block | 1.8 |
| `skip` | off, on | a skip connection around the hidden block | |
| `dropout` | 0 to 0.5 | regularisation | 1.8 |
| `sigma` | 0 to 0.5 | regularisation by noise on the input vectors | 1.8 |
| `lr` | 1e-4 to 1e-2 | the step the optimizer takes | 1.4 |

**The next lesson discusses these goals on the class's runs.** Everyone's trials are put
side by side, and we go through them together, axis by axis. Before then, make
sure you understand what your own search shows: for each axis, what did it do to accuracy
and to `train_seconds`, and does that match what the goal says it should? Where it does
not, that is the most interesting thing to bring.

Width, dropout, noise and the learning rate are continuous: a trial draws a number from a
range, not one of a few values. It is someone's design, not the only one; in Part 2 you
design your own. Here, run it as it is, so that your search can be compared with everyone
else's.

It is not seeded, and it samples at random: every trial is an independent draw from the
space. Your 24 trials are other draws than your neighbour's, started from other weights, so
your winner may well not be theirs. That is the point of looking at the class's runs
together. One search is one draw; fifteen show which choices win every time and which only
won once. And because no trial depends on the ones before it, the class's trials pooled are
one random search of a few hundred trials, dense enough to see a trend along a continuous
axis that 24 points cannot show.

Notebook 04 searched with TPE, which samples where earlier trials scored well. That spends
a small budget better, but it makes the trials depend on each other: fifteen TPE searches
each crowd around their own first lucky draws, and pooled they are not a sample of the
space. `search` takes the sampler as an argument; Part 2 may use either.

`objective` and `search` are the same machinery Part 2 uses. `objective` draws a config
from the space, trains it, and reports its accuracy to Optuna after every epoch; `search`
runs `N_TRIALS` trials of a space as one Optuna study.

Because a trial reports every epoch, a study can have a **pruner**: it stops a trial
whose accuracy after some epochs is below what earlier trials had at that epoch, and
spends the time on the next one. The handed-out search runs without a pruner. A pruned
trial has no final accuracy, and the class's pooled sample needs every draw to count,
the bad ones too. Part 2 prunes.

In [ ]:
USED = {*HEAD, "dataset", "encoder", "epochs", "n_train"}


def objective(space: Space, log: StudyLog):
    """What an Optuna study calls once per trial: draw a config from `space`, train it,
    report the accuracy after every epoch, and return the last one. Every trial logs
    every metric, the accuracy is the one the search maximises."""
    unused = sorted(set(space) - USED)
    if unused:
        raise ValueError(
            f"objective does not use {unused}: build it in make_head (and give "
            "it a default in HEAD), or take it out of the space"
        )

    def run_trial(trial: optuna.Trial) -> float:
        config = space.draw(trial)
        tr, te = loaders(
            config["dataset"], config["encoder"], n_train=config.get("n_train")
        )
        dim, n_classes = shape_of(config["dataset"], config["encoder"])
        model = make_head(config, dim, n_classes)
        seen = None

        # report every epoch, not only the last: the trial's learning curve reaches the
        # store, and the study's pruner can stop a trial that is behind the others
        def to_optuna(epoch, fit):
            nonlocal seen
            seen = fit
            trial.report(fit.scores().accuracy, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        try:
            training = Training(
                lr=config["lr"], epochs=config["epochs"], logdir=STORE.logdir()
            )
            seen = training.fit(model, tr, te, classes_of(te), to_optuna)
            return seen.scores().accuracy
        finally:
            # a pruned trial is recorded as well, with the epochs it ran and every
            # metric of its last epoch
            log.record(
                trial,
                RunRecord(
                    role="trial",
                    dataset=config["dataset"],
                    config=config,
                    fixed=space.fixed(),
                    train_seconds=seen.seconds if seen else 0.0,
                    curves=seen.curves if seen else None,
                    scores=seen.scores() if seen and seen.confusion else None,
                    confusion=seen.confusion if seen else None,
                    extra={"embed_dim": dim},
                ),
                Architecture.of(model, torch.zeros(2, dim)),
            )

    return run_trial


def search(
    space: Space,
    name: str,
    sampler=None,
    pruner=None,
    attrs=None,
    idea=None,
    n_trials: int = N_TRIALS,
) -> tuple:
    """Run `n_trials` trials of `space` as the Optuna study `name`, and return the study
    with a table of its finished trials (one row per trial). The study carries `idea`.

    `sampler=None` samples every trial at random and independently; pass
    `optuna.samplers.TPESampler()` for TPE. `pruner=None` lets every trial run all its
    epochs. Nothing is seeded: the trials drawn and the weights they start from differ
    per run of the notebook, so two students' searches of one space are two draws from
    it. A study that exists already gets the new trials added to it."""
    study = StudyLog.search(
        name, space, idea, sampler=sampler, pruner=pruner, attrs=attrs
    )
    start = time.time()
    study.optimize(objective(space, study), n_trials)

    done = study.trials(optuna.trial.TrialState.COMPLETE)
    pruned = study.trials(optuna.trial.TrialState.PRUNED)
    table = pd.DataFrame([{"Accuracy": t.value, **t.params} for t in done])
    print(
        f"{n_trials} trials in {time.time() - start:.0f}s; the study {name!r} now holds "
        f"{len(done)} finished and {len(pruned)} pruned\n"
    )
    print(table.sort_values("Accuracy", ascending=False).head(6).to_string(index=False))
    return study, table


# The search space handed out to the whole class, searched at random. Part 2 starts from
# a copy of it. Both ends of a range are included.
HANDED_OUT_SPACE = Space(
    {
        "dataset": DATASET,
        "encoder": Choice(["dinov2-small"]),
        "n_hidden_layers": Int(0, 3),
        "hidden_dim": Int(32, 1024, log=True),  # built only when n_hidden_layers > 0
        "activation": Choice(["relu", "gelu", "tanh", "sigmoid"]),
        "norm": Choice(["none", "batch", "layer"]),
        "skip": Choice([False, True]),
        "dropout": Float(0.0, 0.5),
        "sigma": Float(0.0, 0.5),
        "lr": Float(1e-4, 1e-2, log=True),
        "epochs": EPOCHS,
    }
)

In [ ]:
handed_out, df = search(
    HANDED_OUT_SPACE, "lesson1-notebook", attrs={"origin": "notebook"}, idea=HANDED_OUT
)
print(f"\nbest trial : {df['Accuracy'].max():.4f}")
print(f"baseline   : {baseline.mean():.4f} +/- {baseline.std():.4f}")

In [ ]:
from plotting import search_parallel_plot

# Same plot as notebook 04 §4.2, generalised to whatever columns a search space produces --
# one line per trial, every axis at once, so an interaction between two axes is still
# visible even though no single-axis panel would show it.
fig = search_parallel_plot(df, columns=list(df.columns), color="Accuracy")
fig.update_layout(title="The handed-out search, every trial and every axis at once")
fig.show()

**Did anything beat the baseline? By how much?**

Careful with the comparison you just printed. The best trial is a **single run**, selected
as the maximum of 24 draws. The baseline is a **mean over several runs**. Taking the max
of many noisy numbers and comparing it to an average is one of the most common ways to
manufacture a win that is not there — the max of 24 draws from the baseline's own
distribution would already sit above the baseline's mean.

Which is exactly what E4 is for.

## E4 — Rerun the winner honestly

Take the winning configuration and rerun it a few times. Then compare like with like.

In [ ]:
best = handed_out.best()
print("winning config:", {k: v for k, v in best.items() if k != "epochs"})
best_dim, _ = shape_of(best["dataset"], best["encoder"])


def make_winner():
    return make_head(best, best_dim, N_CLASSES)


win_tr, win_te = loaders(best["dataset"], best["encoder"])
winner_runs = run_spread(make_winner, win_tr, win_te, lr=best["lr"])
winner = winner_runs.values("accuracy")

print()
compare("baseline", baseline)
compare("search winner", winner, versus=baseline)
print(f"\nsingle-run best from the search : {df['Accuracy'].max():.4f}")
print(f"the same config over {REPEATS} runs : {winner.mean():.4f}")

In [ ]:
# The number that stands for this search: the winner rerun a few times, not its single-run
# leaderboard score. It sits in `lesson1-notebook-runs`, next to the baseline.
RUNS.log(
    "winner",
    "final",
    winner_runs,
    dataset=best["dataset"],
    params=run_params(best["dataset"], best["encoder"], best),
    architecture=Architecture.of(make_winner(), torch.zeros(2, best_dim)),
    attrs={"search_space": HANDED_OUT_SPACE.line()},
)

**This is the number that stands for the search**, not the leaderboard number.

Answer these for yourself before moving on:

1. How much of the apparent win survived being rerun? If it shrank, by how much?
2. Is the difference larger than the pooled sigma printed above? If not, what is the honest
   one-sentence conclusion?
3. Most "wins" at this scale are inside noise between runs. Is this one? Commit to an answer.

## E5 — An ablation you can actually settle

`GaussianNoise` is the one head-side knob with any prior evidence behind it: earlier
measurements had it worth about +0.4 points on DTD. That is small enough that it is a
genuinely open question rather than a known result.

**Predict first:** does input noise still help once the training set is full size? Argue
from what noise is *for* — then test it at two training sizes.

In [ ]:
FULL = 3760


def with_noise(sigma):
    return lambda: Serial(
        [GaussianNoise(sigma=sigma), nn.Linear(DIM["dinov2-small"], N_CLASSES)]
    )


def train_accuracy(model, train_dl) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in train_dl:
            correct += int((model(X).argmax(1) == y.argmax(1)).sum())
            total += len(y)
    return correct / total


for n_train in (param(940, test=200), None):
    tr, te = loaders(DATASET, "dinov2-small", n_train=n_train)
    rows = n_train or FULL
    # scale epochs so every training size gets a comparable number of gradient steps,
    # otherwise you are measuring "less data" and "less training" at the same time
    epochs = max(EPOCHS, round(EPOCHS * FULL / rows))

    print(f"\n--- {rows} rows, {epochs} epochs ---")
    clean_runs = run_spread(with_noise(0.0), tr, te, epochs=epochs)
    noisy_runs = run_spread(with_noise(0.3), tr, te, epochs=epochs)
    clean, noisy = clean_runs.values("accuracy"), noisy_runs.values("accuracy")
    compare("sigma = 0.0", clean)
    compare("sigma = 0.3", noisy, versus=clean)
    for sigma, runs in ((0.0, clean_runs), (0.3, noisy_runs)):
        RUNS.log(
            f"noise-{sigma}-{rows}rows",
            "ablation",
            runs,
            dataset=DATASET,
            params=run_params(
                DATASET, "dinov2-small", {"sigma": sigma}, n_train=rows, epochs=epochs
            ),
            architecture=Architecture.of(
                with_noise(sigma)(), torch.zeros(2, DIM["dinov2-small"])
            ),
        )

    fitted = with_noise(0.0)()
    run(fitted, tr, te, epochs=epochs)
    print(f"{'training-set accuracy':<32} {train_accuracy(fitted, tr):.4f}")

Note the epoch scaling in that cell. Without it you would be changing two things at once —
less data *and* fewer gradient steps — and could not attribute the difference to either.
Was that the right call?

Three things to take from the output, and the third is the one worth keeping.

**Check whether either difference cleared its sigma.** If it did not, the honest sentence is
"no measurable effect at this training size", not "noise helped slightly". Resist the pull
of the sign: a +0.1 pt difference at 0.4 sigma is not a small benefit, it is no evidence.

**"Does GaussianNoise help?" is not a question with an answer.** "Does it help at *this*
training size, on *this* representation?" is. Regularisation is supposed to buy something
exactly when a model can memorise its training set — so look at the last line.

**The head reaches ~100% training accuracy and the test number does not move.** That is the
informative result. The model is memorising completely, and regularising it harder changes
nothing downstream, which means the ceiling you are hitting is not head capacity — it is
what the frozen representation makes linearly separable. You cannot regularise your way to
a better representation. Notebook 04 reached the same conclusion from the other direction:
the encoder moves accuracy by more than ten points, the head knobs by about two.

That is the end of the handed-out part. Everything it stored is in two Optuna
studies: `lesson1-notebook`, the trials of the search, and `lesson1-notebook-runs`, the
baseline, the encoder comparison, the winner and the noise ablation.

# Part 2 — your own study

One study of your own: an idea you want to test, a search space designed to test it, and
the same discipline as Part 1 — a baseline, a search, the winner rerun a few times, and the
difference in sigmas. Some ideas that are cheap and genuinely open:

- Does the encoder ranking from E2 hold on a *different* dataset (`eurosat`), or is it a
  property of textures specifically?
- Does fusion (notebook 04 §4.3) behave differently when the two encoders are close in
  strength — say `resnet-18` + `mobilenet_v2` — than when one clearly dominates?
- Does the gap between the encoders widen or narrow as you shrink the training set? Which
  direction do you expect, and why?
- One hidden layer beats the linear baseline by about two points on DTD. Does that gain hold
  on EuroSAT, where the linear head already scores around 0.94?
- Width and depth of the head (`hidden_dim`, `n_hidden_layers`) against accuracy, and against
  what they cost: `train_seconds` and `n_params`. Notebook 04 found added depth losing on a
  frozen encoder; where does more width stop paying?
- Normalisation in the hidden block (`norm`: `"batch"`, `"layer"` or `"none"`). The vectors
  come out of a frozen encoder already on a fixed scale; does normalising inside the head
  still buy anything?
- A skip connection around the hidden block (`skip`), vectormesh's `Skip`: it
  layer-normalises its input and adds a projection of it to the block's output, so a skip
  study also changes the normalisation and `n_params`. Compare it against `norm="layer"` to
  tell the two apart. With one or two hidden layers there is little depth to skip; does it
  change anything here?
- A gate on the hidden block (`gate`), vectormesh's `Gate`: `sigmoid(W·h) * h`. Gating has so
  far tied its capacity-matched control in this course's measurements; is that true for
  your head too?

**Read as a parallel-coordinates plot.** optuna-dashboard draws a study as one
parallel-coordinates plot: the params a trial drew on the axes (`encoder`,
`n_hidden_layers`, `hidden_dim`, `activation`, `norm`, `dropout`, `skip`, `gate`, `lr`,
...) and the accuracy at the end. A study that varies one switch with everything else
equal is the easiest to read there: its lines differ on one axis, so whatever the
accuracy does, that axis did it.

**Prepare for the lesson.** The next lesson discusses the class's runs live, and
for your study you will be asked three things:

1. **Why did you do this?** What made this the question worth your budget?
2. **What was the idea behind it?** What did you expect to happen, and from what?
3. **What did you learn when it disappointed?** A low accuracy can be the interesting
   result: a search that found nothing, or a winner that shrank back into the baseline's
   noise once rerun, says something about the representation, the space or the idea.

Think these through for yourself, before you run and after. Nothing is written down or
handed in: the runs are what you bring, and the answers are what you say about them.

A study over a design of your own, outside this notebook, follows the loop in [`exercises/readme.html`](exercises/readme.html).

**Name your study.** Part 2 is a study of your own, and its name is the name of its
Optuna study, which is how you find it in the dashboard. Name it **your first name, a
dash, and what it studies**, in lowercase letters, digits and dashes:
`sam-depth-on-eurosat`, `noor-fusion-resnet-mobilenet`. The first name says whose study
it is, the rest what it is about. The name is asked here because everything below uses
it. While it holds its placeholder, the notebook stops at this cell, before anything of
your own study is stored.

In [ ]:
STUDY = study_name("name-your-study")

**Then say your idea.** An idea is what your study investigates, in a few words: "does
depth still help on eurosat". Its name is made from your GitHub login, those words and
the ideas it builds on, and reads like `blissful-bee-18`. A classmate builds on your
idea by passing that name as `builds_on`, and you build on theirs the same way: the
names of the class's ideas are on the class page (`make dashboard LESSON=1`). Your first
idea builds on Part 1, the idea the whole class ran: `HANDED_OUT`. An idea
you build on must be published. Other words are another idea, and another idea is
another study, so it needs another `STUDY` name. The placeholder stops the notebook
here, as the study's name does.

In [ ]:
IDEA = Idea.of(
    "say what your study investigates",  # in your own words, a few of them
    # the idea you build on: the exercise as handed out, or the name of an idea of the
    # class, as the class page shows it: "blissful-bee-18"
    builds_on=HANDED_OUT.name,
)
print(f"idea : {IDEA}")

**Then design the space**, in the cell below. It starts as a copy of the handed-out one;
every line you keep is a decision too. Decide these before you run anything:

- **Should `encoder` be in the space at all?** You have measured that axis directly in E2.
  Leaving it in means most of your budget gets spent re-confirming what you already know.
  Taking it out means your search can only explore the flat part. Which is the better use
  of the trials your budget holds? There is a defensible answer either way — pick one and know why.
- **What is the widest hidden layer worth trying?** Notebook 04's evidence says added depth
  loses on a frozen encoder. Is your space designed to *find* a good architecture, or to
  *test* that idea? Those want different ranges.
- **How wide a learning-rate range?** Too narrow and you have assumed the answer; too wide
  and you burn trials on configs that cannot train.
- **Which dataset?** `dataset` is an axis like any other; every name in `CACHES` works, with
  any encoder listed for it there.

That is the whole knob-set the handed-out space uses, and it is not the only one. A few
ways to widen the design space, none required, all cheap to try:

- **A `Parallel` fusion branch**, notebook 04 §4.3's construction, as one more entry in the
  space instead of a separate experiment — does giving the search the *option* of fusion
  change what it picks, compared to deciding by hand? This one needs code of your own: a
  joined loader as in 04, and a branch in `make_head` and `trainable`.
- **The size of the training set** (`n_train`, a number of rows): `trainable` passes it to
  `loaders`, which takes a shuffled subset. Does the gap between your options widen or narrow
  with less data?
- **More encoders than the ones published here.** Every one you have used so far was
  chosen for you. If you want to try another — a CLIP variant, a newer DINOv3, a
  domain-specific encoder — an LLM with web or an MCP tool for browsing the HuggingFace Hub
  can shortlist candidates faster than reading model cards one by one; a defensible
  rationale still has to be *yours* (what was it pretrained on, and does that resemble
  textures?). Once you have picked one, `01c_build_your_own_cache.ipynb` (optional, in this
  same folder) builds a small cache for it on any hardware; add it to `CACHES`.
- **A different kind of head knob entirely**, one that `make_head` does not build yet:
  weight decay, label smoothing, a different optimizer, a wider first layer than the
  rest. The space
  you were handed was itself a choice, not a law. A new axis needs a line in `make_head`
  (or in `trainable`, for what goes to the optimizer) that uses it, and a default in
  `HEAD`.
- **The search algorithm.** `search(..., sampler=optuna.samplers.TPESampler())` searches with TPE
  instead of at random. Over a small budget it may find a better winner; its trials are no
  longer a sample of the space, so read its per-axis tables with that in mind.

- **The pruner.** The cell that runs your search gives it a `PercentilePruner`: it stops
  a trial that scores below the lowest quarter of the trials that ran to the end. Pruned
  trials do not count in that comparison, so the bar rises while the search runs, and
  it stops more than a quarter of them; the first 5 trials always run in full, so a
  search of 5 trials or fewer prunes nothing. Read in the dashboard which trials
  it stopped and after how many epochs; would any of them have caught up?
  `MedianPruner()` is stricter, `pruner=None` lets every trial finish.

Your space travels with the study. Optuna stores, for every trial, the distribution each
param was drawn from, and the study carries the whole space on one line as its user attr
`search_space`, next to `searched_axes`, the axes you searched.

In [ ]:
# ---- YOUR SEARCH SPACE. It starts as a copy of the handed-out one; make it test your idea. ----
param_space = Space(
    {
        "dataset": "dtd",
        "encoder": Choice(["dinov2-small"]),
        "n_hidden_layers": Int(0, 3),
        "hidden_dim": Int(32, 1024, log=True),  # built only when n_hidden_layers > 0
        "activation": Choice(["relu", "gelu", "tanh", "sigmoid"]),
        "norm": Choice(["none", "batch", "layer"]),
        "skip": Choice([False, True]),
        "dropout": Float(0.0, 0.5),
        "sigma": Float(0.0, 0.5),
        "lr": Float(1e-4, 1e-2, log=True),
        "epochs": EPOCHS,
        # one more switch for the hidden block; make_head builds it, HEAD holds its default
        # "gate": Choice([True, False]),
    }
)

**Then give the study its budget.** The budget is yours: the minutes you give this
study, on the hardware in front of you. Part 1 measured what one trial costs here, and
`Budget.measure` turns your minutes into trials with it, at your `EPOCHS`, after setting aside
the `REPEATS` runs of a baseline, of your winner rerun and of the two arms at the end.
The cost of a trial was measured on DTD with `dinov2-small`: a larger dataset or encoder
costs more per trial, so give such a study more minutes than you count on.

Read the last line it prints. A few trials per searched axis cannot tell a good range
from a lucky draw, so fix axes to one value or drop them until the ones left get
enough. Which ones you fix is the decision goal 1.14 asks you to defend. The class
page pools everyone's handed-out search: an axis that is flat there is the first to
fix. And goal 1.10: whatever
your search finds, a win is read against the spread between runs and against what it
cost.

In [ ]:
MINUTES = 15  # the time you give this study on this machine
BUDGET = Budget.measure(
    MINUTES,
    StudyLog.load("lesson1-notebook"),
    epochs=EPOCHS,
    repeats=REPEATS,
    reruns=4,  # the baseline, the winner and the two arms
    space=param_space,
)
MY_TRIALS = param(BUDGET.n_trials, test=2)
print(
    f"one trial      : {BUDGET.seconds_per_trial}s at {EPOCHS} epochs, "
    f"as measured in Part 1 ({BUDGET.measured_on})"
)
print(
    f"trials that fit: {BUDGET.n_trials} in {MINUTES} minutes, after the baseline, "
    f"the winner and the two arms ({REPEATS} runs each)"
)
print(
    f"per axis       : {BUDGET.trials_per_axis} trials for each of the "
    f"{BUDGET.searched_axes} axes you search"
)
if BUDGET.n_trials == 0:
    raise ValueError(
        f"budget: {MINUTES} minutes hold no trial on this machine once the baseline, "
        "the winner and the two arms are set aside; give the study more minutes or fewer epochs"
    )

In [ ]:
STUDY_NAME = f"lesson1-study-{STUDY}"
study_attrs = {
    "origin": "study",
    "study": STUDY,
    "budget": {**BUDGET.model_dump(), "n_trials": MY_TRIALS},
    "space_is_default": param_space == HANDED_OUT_SPACE,
}
print(f"study    : {STUDY_NAME}")
print(f"space    : {param_space.line()}")
print(f"same space as handed out: {study_attrs['space_is_default']}")

In [ ]:
# Random sampling, as in the handed-out search; sampler=optuna.samplers.TPESampler() for
# TPE. The pruner lets the first 5 trials run in full; after that it stops a trial that,
# at its fifth epoch or later, scores below the lowest quarter of the trials that ran to
# the end, at that epoch (75.0: the top 75% go on).
mine, my_df = search(
    param_space,
    STUDY_NAME,
    pruner=optuna.pruners.PercentilePruner(75.0, n_startup_trials=5, n_warmup_steps=4),
    attrs=study_attrs,
    idea=IDEA,
    n_trials=MY_TRIALS,
)
# the runs of your study that are given their params: baseline, winner, the two arms
my_runs = StudyLog.runs(f"{STUDY_NAME}-runs", IDEA, choices=CHOICES)

**Rerun your winner a few times, against a baseline on the same dataset.** When the winner
is on DTD, the baseline is the one from E1; on another dataset, the cell measures the same
linear head on `dinov2-small` there first. Both are stored with your study, in
`lesson1-study-<your study>-runs`.

In [ ]:
my_best = mine.best()
my_dataset = my_best["dataset"]
print("winning config:", {k: v for k, v in my_best.items() if k != "epochs"})

base_dim, base_classes = shape_of(my_dataset, "dinov2-small")
if my_dataset == DATASET:
    my_baseline_runs = baseline_runs
else:
    base_tr, base_te = loaders(my_dataset, "dinov2-small")
    my_baseline_runs = run_spread(
        lambda: nn.Linear(base_dim, base_classes), base_tr, base_te
    )
my_baseline = my_baseline_runs.values("accuracy")
my_runs.log(
    "baseline",
    "baseline",
    my_baseline_runs,
    dataset=my_dataset,
    params=run_params(my_dataset, "dinov2-small"),
    architecture=Architecture.of(
        nn.Linear(base_dim, base_classes), torch.zeros(2, base_dim)
    ),
)

my_dim, my_classes = shape_of(my_dataset, my_best["encoder"])


def make_my_winner():
    return make_head(my_best, my_dim, my_classes)


my_tr, my_te = loaders(my_dataset, my_best["encoder"])
my_winner_runs = run_spread(make_my_winner, my_tr, my_te, lr=my_best["lr"])
my_winner = my_winner_runs.values("accuracy")
my_runs.log(
    "winner",
    "final",
    my_winner_runs,
    dataset=my_dataset,
    params=run_params(my_dataset, my_best["encoder"], my_best),
    architecture=Architecture.of(make_my_winner(), torch.zeros(2, my_dim)),
)

print()
compare(f"baseline on {my_dataset}", my_baseline)
compare("your search winner", my_winner, versus=my_baseline)
print(f"\nsingle-run best from your search : {my_df['Accuracy'].max():.4f}")

**Read it in sigmas, not in points**: inside about two, the difference is not a result,
whatever its sign. If it disappointed, that is the third question from above, and the runs
reach the class as they are.

## A second arm of your own

A search answers "what is the best configuration in this space". Many ideas are sharper
than that: *this* change against *that* control, everything else equal. The cell below
builds both arms from the helpers in this notebook, runs each `REPEATS` times, and stores
them with your study under the role `hypothesis`.

As handed out it tests one of the ideas above: one hidden layer against the linear head on
EuroSAT. Left like that it repeats a comparison the whole class runs, so change it to your
own: set `ARM_DATASET`, `ARM_ENCODER`, `control` and `treatment` to the comparison your
idea needs; `make_head` builds any head the search can, and `loaders` and
`shape_of` find any cache in `CACHES`.

In [ ]:
ARM_DATASET, ARM_ENCODER = "eurosat", "dinov2-small"
control = {"n_hidden_layers": 0, "hidden_dim": 0, "sigma": 0.0, "lr": 1e-3}
treatment = {"n_hidden_layers": 1, "hidden_dim": 256, "sigma": 0.0, "lr": 1e-3}


def arm(name: str, config: dict) -> np.ndarray:
    """Run one arm REPEATS times and store it with the study as a hypothesis run."""
    dim, n_classes = shape_of(ARM_DATASET, ARM_ENCODER)
    tr, te = loaders(ARM_DATASET, ARM_ENCODER)
    runs = run_spread(
        lambda: make_head(config, dim, n_classes), tr, te, lr=config["lr"]
    )
    my_runs.log(
        f"hypothesis-{name}",
        "hypothesis",
        runs,
        dataset=ARM_DATASET,
        params=run_params(ARM_DATASET, ARM_ENCODER, config, arm=name),
        architecture=Architecture.of(
            make_head(config, dim, n_classes), torch.zeros(2, dim)
        ),
    )
    return runs.values("accuracy")


control_scores = arm("control", control)
treatment_scores = arm("treatment", treatment)
compare(f"control   ({ARM_DATASET})", control_scores)
compare(f"treatment ({ARM_DATASET})", treatment_scores, versus=control_scores)

---

## Read your search in optuna-dashboard

Every cell above wrote to `artefacts/optuna/studies.db`. From the root of the course
repository:

```bash
make optuna-dashboard
```

and open http://127.0.0.1:8088. Four studies are there:

| study | what it holds |
|---|---|
| `lesson1-notebook` | the trials of the handed-out search |
| `lesson1-notebook-runs` | the baseline, the encoder comparison, the winner rerun a few times, the noise ablation |
| `lesson1-study-<your study>` | the trials of your own search, the pruned ones included |
| `lesson1-study-<your study>-runs` | your baseline, your winner, your two arms |

Open a study; its pages are in the bar on the left:

- **History**: the accuracy of every trial in the order they ran with the best so far,
  which axis explains most of the differences in accuracy (**Hyperparameter
  Importance**; with a few dozen trials a hint, to check against the baseline's spread between runs), and
  **Intermediate values**: every trial's accuracy per epoch, one line per trial. In your
  own study the pruned trials stop early; in a `-runs` study the steps are the repeats of
  one run, so the height of a line's wiggle is its spread between runs.
- **Analytics**: accuracy against one param (Slice) and against two (Contour, Rank).
- **Smart Selection**: the parallel-coordinates plot. The params are on the axes, and the
  numeric user attrs `n_params` and `train_seconds` can be ticked as axes too. Drag along
  an axis to keep the trials in that range; the history plot and the table below follow.
- **Trials (List)**: one page per trial, with a note that holds the layer table and the
  tensor shape at every layer, the accuracy per epoch, the params, the user attrs (`role`,
  `dataset`, `n_params`, `architecture`, `train_seconds`, the loss curves) and the
  artifact `architecture.json`.

The study's own page shows its user attrs: its idea (`idea`, the words, and `idea_name`,
the name a classmate builds on, next to `idea_author`, `idea_parents` and `idea_id`), and
for a search `search_space` and `searched_axes`. Your search adds `study`,
`space_is_default` and `budget`: the minutes you gave it, the seconds per trial measured
in Part 1, and `n_trials`, the trials it ran.

## Hand in your runs before the next lesson

Every cell above wrote its runs to the Optuna store: Part 1 in `lesson1-notebook` and
`lesson1-notebook-runs`, Part 2 in `lesson1-study-<your study>` and
`lesson1-study-<your study>-runs`. Each study carries its idea, each trial your study's name and search
space, and its architecture with the tensor shape at every layer.

Send them to the class runs repository. From the root of the course repository, first look
at what would be handed in:

```bash
make share LESSON=1 ARGS=--list
```

Then hand them in:

```bash
make share LESSON=1
```

Or tell your coding agent: *"follow docs/submit-runs.md for lesson 1"*. That file walks it
through the one-time setup and the submission. Handing in again later is safe; only new
runs are added.

The next lesson opens on the whole class's runs: everyone's handed-out runs side by
side (they should agree, up to the spread between runs and between machines), and every
study with its search space, its baseline and its final. Your runs are part of that
picture, including the ones that lost. To look at the class's runs yourself:
`make dashboard LESSON=1`.
